# 📖 Manhwa / Webtoon English → Arabic Translator

Professional-grade pipeline: detection → OCR → page-level context → LLM translation →
text removal & inpainting → Arabic RTL typesetting → QA.

1. **Runtime → Change runtime type → GPU (T4)** before running.
2. Run the cells in order; the last cell starts the Gradio GUI.
3. For best translation quality, set an OpenAI-compatible LLM API key in the Settings tab
   (e.g. Gemini: `https://generativelanguage.googleapis.com/v1beta/openai` with model `gemini-2.0-flash`).
   Without a key the app falls back to an offline MarianMT model.

In [ ]:
!nvidia-smi || echo "No GPU detected — CPU mode (slower)"
%cd /content
!rm -rf ManhwaArabicTranslator || true
# Upload the project (zip of manhwa_arabic_translator/) or clone it:
# !git clone <your-repo> ManhwaArabicTranslator
import os
if not os.path.isdir('ManhwaArabicTranslator'):
    from google.colab import files
    print('Upload the zip of the project (manhwa_arabic_translator folder zipped):')
    up = files.upload()
    import zipfile, glob
    zp = list(up.keys())[0]
    with zipfile.ZipFile(zp) as z: z.extractall('ManhwaArabicTranslator')
%cd /content/ManhwaArabicTranslator/manhwa_arabic_translator 2>/dev/null || %cd /content/ManhwaArabicTranslator
print('Project ready at', os.getcwd())

In [ ]:
%%time
# Install dependencies. NOTE: gradio must be >=5 — gradio 4.x is
# incompatible with the new huggingface_hub preinstalled on Colab.
!pip install -q -U 'gradio>=5'
!pip install -q -r requirements.txt
!pip install -q --no-deps simple-lama-inpainting || echo "LaMa optional — OpenCV fallback will be used"

In [ ]:
import os
# Optional: set your LLM credentials here (or later in the GUI Settings tab)
# os.environ['MANHWA_LLM_BASE_URL'] = 'https://generativelanguage.googleapis.com/v1beta/openai'
# os.environ['MANHWA_LLM_API_KEY']  = 'YOUR_API_KEY'
# os.environ['MANHWA_LLM_MODEL']    = 'gemini-2.0-flash'
print('Env set:', {k: v[:8] + '...' for k, v in os.environ.items() if k.startswith('MANHWA_')})

In [ ]:
%%time
# Warm up: download fonts + OCR models so the first translation is fast
# IMPORTANT: delete any old font cache — older builds downloaded Cairo,
# which misses many Arabic presentation forms (tofu glyphs).
!rm -rf fonts
import sys; sys.path.insert(0, '.')
from app.config import AppConfig, setup_logging
setup_logging()
cfg = AppConfig.load()
from app.typography.fonts import ensure_fonts
fonts = ensure_fonts('fonts')
print('Fonts:', list(fonts))
from app.models.manager import detect_device
print('Device:', detect_device())
# Pre-load EasyOCR (downloads detection/recogniton weights on first use)
from app.ocr.easyocr_engine import EasyOCREngine
EasyOCREngine().reader()
print('OCR ready')

In [ ]:
# 🚀 Launch the GUI — errors are shown directly; open the printed gradio.live link
# NOTE: this cell keeps running (that's normal). To stop it: Runtime → Interrupt execution.
import os, sys, traceback
os.environ['MANHWA_PUBLIC'] = '1'
sys.path.insert(0, os.getcwd())          # make sure `app` package is importable
print('Working dir:', os.getcwd())
print('app/ folder found:', os.path.isdir('app'))
assert os.path.isdir('app'), 'app/ not found — re-run the setup cells above (the %cd cell)!'

from app.config import AppConfig, setup_logging
setup_logging()
cfg = AppConfig.load()

from app.ui.gradio_app import GradioApp
ui = GradioApp(cfg, workdir='.')
demo = ui.build()

# try several ports in case an older run still holds 7860
launched = False
for port in (7860, 7861, 7862, 7863):
    try:
        demo.queue(default_concurrency_limit=1).launch(
            server_name='0.0.0.0', server_port=port,
            share=True,          # creates the https://xxxx.gradio.live link
            show_error=True)
        launched = True
        break
    except Exception:
        print(f'--- launch on port {port} failed ---')
        traceback.print_exc()

if not launched:
    # last resort: Colab's own proxy (works even if the gradio.live tunnel is blocked)
    demo.queue(default_concurrency_limit=1).launch(
        server_name='0.0.0.0', server_port=7865, share=False, show_error=True)
    from google.colab.output import eval_js
    print('⇒ Open this URL instead:', eval_js('google.colab.kernel.proxyPort(7865)'))